In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [ ]:

# Step 1: Define stock
tickers = ["RELIANCE.NS"]

for ticker in tickers:
    print(f"Processing {ticker}...")

    # Step 2: Download data
    data = yf.download(
        ticker,
        period="6mo",
        interval="1d"
    )

    # Step 3: Log Returns
    data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))

    # Step 4: Turnover (₹ traded)
    data["Turnover"] = data["Close"] * data["Volume"]

    # Step 5: Average Turnover
    avg_turnover = data["Turnover"].mean()

    # Step 6: Turnover Ratio (normalized)
    data["Turnover Ratio"] = data["Turnover"] / avg_turnover

    # Step 7: Amihud Illiquidity
    data["Amihud"] = abs(data["Log Return"]) / data["Turnover"]

    # Step 8: Clean data
    data = data.replace([np.inf, -np.inf], np.nan).dropna()

    # Step 9: Output
    print(data[["Close", "Volume", "Log Return", "Turnover", "Turnover Ratio", "Amihud"]].head())

    # Step 10: Save
    data.to_excel(f"{ticker}_full_liquidity_analysis.xlsx")

    # Rolling 20-day volatility
    data["Rolling Volatility (20D)"] = data["Log Return"].rolling(20).std()

    # Annualized volatility
    data["Annualized Volatility"] = data["Rolling Volatility (20D)"] * (252 ** 0.5)

    # Clean
    data = data.dropna()

    print(data[["Log Return", "Rolling Volatility (20D)", "Annualized Volatility"]].head())

    # Save
    data.to_excel("volatility_analysis.xlsx")

In [ ]:

# Step 1: Load data
ticker = "RELIANCE.NS"

data = yf.download(ticker, period="6mo", interval="1d")

# Step 2: Compute metrics
data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))
data["Turnover"] = data["Close"] * data["Volume"]
data["Amihud"] = abs(data["Log Return"]) / data["Turnover"]

data = data.replace([np.inf, -np.inf], np.nan).dropna()

# Step 3: Create plots
plt.figure(figsize=(12, 10))

# Plot 1: Log Returns
plt.subplot(3, 1, 1)
plt.plot(data.index, data["Log Return"])
plt.title(f"{ticker} - Daily Log Returns")
plt.xlabel("Date")
plt.ylabel("Log Return")

# Plot 2: Daily Turnover
plt.subplot(3, 1, 2)
plt.plot(data.index, data["Turnover"])
plt.title(f"{ticker} - Daily Turnover")
plt.xlabel("Date")
plt.ylabel("Turnover")

# Plot 3: Amihud Illiquidity
plt.subplot(3, 1, 3)
plt.plot(data.index, data["Amihud"])
plt.title(f"{ticker} - Amihud Illiquidity")
plt.xlabel("Date")
plt.ylabel("Amihud")

# Layout fix
plt.tight_layout()

# Show graph
plt.show()

In [ ]:


# Step 1: Load tickers
file_path = "ind_nifty50list.xlsx"
df = pd.read_excel(file_path)

tickers = [symbol + ".NS" for symbol in df["Symbol"].tolist()]

# Step 2: Store results
avg_illiquidity = []

# Step 3: Loop
for ticker in tickers:
    print(f"Processing {ticker}...")

    data = yf.download(ticker, period="6mo", interval="1d")

    if data.empty:
        avg_illiquidity.append(None)
        continue

    # Step 4: Log returns
    data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))

    # Step 5: Turnover
    data["Turnover"] = data["Close"] * data["Volume"]

    # Step 6: Amihud Illiquidity
    data["Amihud"] = abs(data["Log Return"]) / data["Turnover"]

    # Clean NaNs / inf
    data = data.replace([np.inf, -np.inf], np.nan).dropna()

    # Step 7: Average Amihud
    avg_illiquidity.append(data["Amihud"].mean())

# Step 8: Add to Excel
df["Average Amihud Illiquidity"] = avg_illiquidity

# Step 9: Save file
output_file = "nifty50_with_illiquidity.xlsx"
df.to_excel(output_file, index=False)

print("File saved as:", output_file)

In [ ]:

ticker = "RELIANCE.NS"

# Step 1: Download data
data = yf.download(ticker, period="6mo", interval="1d")

# Step 2: Core calculations
data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))

data["Turnover"] = data["Close"] * data["Volume"]

data["Amihud"] = abs(data["Log Return"]) / data["Turnover"]

# Rolling volatility (annualized)
data["Volatility"] = data["Log Return"].rolling(20).std() * (252 ** 0.5)

# Squared returns (for clustering)
data["Squared Return"] = data["Log Return"] ** 2

# Step 3: Clean AFTER all calculations
data = data.replace([np.inf, -np.inf], np.nan).dropna()

#PLOT: Volatility vs Illiquidity
plt.figure(figsize=(8, 6))

plt.scatter(data["Amihud"], data["Volatility"])

plt.xlabel("Amihud Illiquidity")
plt.ylabel("Volatility (Annualized)")
plt.title(f"{ticker} - Volatility vs Illiquidity")

plt.grid(True)
plt.show()

#Correlation
correlation = data["Amihud"].corr(data["Volatility"])
print("Correlation between Amihud and Volatility:", correlation)

In [ ]:

ticker = "RELIANCE.NS"  # change for second stock

data = yf.download(ticker, period="6mo", interval="1d")

# Log returns
data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))

# Squared returns (key for clustering)
data["Squared Return"] = data["Log Return"] ** 2

# Rolling volatility (20-day)
data["Rolling Volatility(annualized)"] = data["Log Return"].rolling(20).std() * (252 ** 0.5)

data = data.dropna()

# 📊 Plot
plt.figure(figsize=(12, 10))

# Log returns
plt.subplot(3, 1, 1)
plt.plot(data.index, data["Log Return"])
plt.title(f"{ticker} - Log Returns")

# Squared returns
plt.subplot(3, 1, 2)
plt.plot(data.index, data["Squared Return"])
plt.title(f"{ticker} - Squared Returns (Volatility Clustering)")

# Rolling volatility
plt.subplot(3, 1, 3)
plt.plot(data.index, data["Rolling Volatility(annualized)"])
plt.title(f"{ticker} - Rolling Volatility (20D)")

plt.tight_layout()
plt.show()